# 🎙️ Voice Cloning API
### Built with XTTS v2 + FastAPI + ngrok

---

## ⚡ Before Running
1. `Runtime → Change runtime type → GPU (T4)`
2. Get free ngrok token at [ngrok.com](https://ngrok.com) → Dashboard → Auth Token
3. Run cells **top to bottom** — or use `Runtime → Run All`

---

## 📋 Cell Overview
| Cell | Purpose |
|------|---------|
| 1 | Install dependencies |
| 2 | Configure ngrok + API key |
| 3 | Create folders |
| 4 | Load XTTS v2 model |
| 5 | Audio preprocessor |
| 6 | Build FastAPI app |
| 7 | Start server + get public URL |
| 8 | Check API status |
| 9 | Test cloning |
| 10 | Get connection details |
| 11 | Save outputs to Google Drive |

---
## Cell 1 — Install Dependencies

In [ ]:
# Install ffmpeg for audio conversion (m4a, mp3 → wav)
!apt-get install ffmpeg -qq

# Install Python packages
!pip install coqui-tts --quiet
!pip install fastapi uvicorn python-multipart httpx --quiet
!pip install pyngrok nest-asyncio --quiet
!pip install librosa soundfile noisereduce --quiet

print('✓ ffmpeg installed')
print('✓ All Python packages installed')

---
## Cell 2 — Configure ngrok + API Key
> ⚠️ Replace both values below before running

In [ ]:
from pyngrok import ngrok
import os

# ⚠️ Paste your ngrok token from ngrok.com/dashboard
NGROK_AUTH_TOKEN = "YOUR_NGROK_TOKEN_HERE"

# ⚠️ Set your own secret API key
API_KEY = "YOUR_SECRET_KEY_HERE"

ngrok.set_auth_token(NGROK_AUTH_TOKEN)
os.environ['API_KEY'] = API_KEY

print(f'✓ ngrok configured')
print(f'✓ API Key set: {API_KEY}')

---
## Cell 3 — Create Folders

In [ ]:
import os

os.makedirs('uploads', exist_ok=True)
os.makedirs('outputs', exist_ok=True)

print('✓ uploads/ folder ready')
print('✓ outputs/ folder ready')

---
## Cell 4 — Load XTTS v2 Model
> ⏳ Downloads ~1.8GB on first run. Takes 2-3 minutes.

In [ ]:
import torch
from TTS.api import TTS

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

if device == 'cpu':
    print('⚠️  No GPU detected!')
    print('   Go to Runtime → Change runtime type → GPU')
    print('   CPU cloning works but takes 3-5 mins per sentence')

print('\nLoading XTTS v2 model...')
tts = TTS('tts_models/multilingual/multi-dataset/xtts_v2').to(device)
print('✓ Model loaded and ready!')

---
## Cell 5 — Audio Preprocessor

In [ ]:
import librosa
import soundfile as sf
import noisereduce as nr
import numpy as np
import subprocess

def convert_to_wav(input_path, output_path):
    """
    Converts any audio format (m4a, mp3, ogg, flac, aac)
    to WAV using ffmpeg.
    """
    subprocess.run([
        'ffmpeg', '-i', input_path,
        '-ar', '22050',   # sample rate XTTS expects
        '-ac', '1',       # mono channel
        '-y',             # overwrite if exists
        output_path
    ], capture_output=True)
    return output_path

def clean_audio(input_path, output_path, target_sr=22050):
    """
    Full audio cleaning pipeline:
    1. Convert to WAV if needed (m4a, mp3, etc)
    2. Reduce background noise
    3. Normalise volume
    4. Trim silence from start and end
    """
    # Convert to WAV first if not already
    if not input_path.lower().endswith('.wav'):
        converted = input_path.rsplit('.', 1)[0] + '_converted.wav'
        convert_to_wav(input_path, converted)
        input_path = converted
        print(f'  ✓ Converted to WAV')

    # Clean pipeline
    audio, sr  = librosa.load(input_path, sr=target_sr)
    reduced    = nr.reduce_noise(y=audio, sr=sr)
    normalised = librosa.util.normalize(reduced)
    trimmed, _ = librosa.effects.trim(normalised, top_db=20)
    sf.write(output_path, trimmed, target_sr)
    return output_path

def score_audio_quality(audio_path):
    """
    Scores reference audio quality 0-100.
    < 40  = Poor   — try a better recording
    40-60 = Fair   — acceptable
    60-80 = Good   — recommended
    > 80  = Excellent — best results
    """
    audio, sr   = librosa.load(audio_path)
    duration    = librosa.get_duration(y=audio, sr=sr)
    signal_pwr  = np.mean(audio**2)
    noise_floor = np.percentile(np.abs(audio), 10)
    snr   = 10 * np.log10(signal_pwr / (noise_floor**2 + 1e-10))
    score = min(100, (duration / 30) * 40 + min(snr, 30) * 2)
    return round(float(score), 1)

print('✓ Audio preprocessor ready')
print('✓ Supported formats: WAV, MP3, M4A, OGG, FLAC, AAC')

---
## Cell 6 — Build FastAPI App

In [ ]:
from fastapi import FastAPI, UploadFile, File, Form, Security, HTTPException
from fastapi.security import APIKeyHeader
from fastapi.responses import FileResponse
import os

app = FastAPI(
    title='Voice Cloning API',
    description='Clone any voice from a short audio sample using XTTS v2',
    version='1.0.0'
)

# ── Security ──────────────────────────────────────────────────
api_key_header = APIKeyHeader(name='X-API-Key', auto_error=False)

def verify_key(key: str = Security(api_key_header)):
    if key is None or key != API_KEY:
        raise HTTPException(
            status_code=403,
            detail='Invalid or missing API key. Pass as X-API-Key header.'
        )
    return key

# ── Root ──────────────────────────────────────────────────────
@app.get('/', tags=['Status'])
def root():
    return {
        'message': 'Voice Cloning API is running',
        'docs': '/docs',
        'health': '/health',
        'endpoints': ['/clone', '/check-audio'],
        'version': '1.0.0'
    }

# ── Health Check ──────────────────────────────────────────────
@app.get('/health', tags=['Status'])
def health():
    return {
        'status': 'running',
        'model': 'XTTS_v2',
        'device': device,
        'gpu_available': torch.cuda.is_available()
    }

# ── Audio Quality Check ───────────────────────────────────────
@app.post('/check-audio', tags=['Utilities'])
async def check_audio(
    reference_audio: UploadFile = File(...),
    api_key: str = Security(verify_key)
):
    """
    Score your reference audio quality before cloning.
    Score above 60 = good. Above 80 = excellent.
    """
    ref_path = f'uploads/check_{reference_audio.filename}'
    with open(ref_path, 'wb') as f:
        f.write(await reference_audio.read())

    # Convert if needed before scoring
    if not ref_path.lower().endswith('.wav'):
        wav_path = ref_path.rsplit('.', 1)[0] + '.wav'
        convert_to_wav(ref_path, wav_path)
        ref_path = wav_path

    score    = score_audio_quality(ref_path)
    audio, sr = librosa.load(ref_path)
    duration = librosa.get_duration(y=audio, sr=sr)
    rating   = 'Poor' if score < 40 else 'Fair' if score < 60 else 'Good' if score < 80 else 'Excellent'

    return {
        'quality_score': score,
        'rating': rating,
        'duration_seconds': round(duration, 2),
        'recommendation': 'Use this audio' if score >= 60 else 'Try a cleaner, longer recording'
    }

# ── Clone Voice ───────────────────────────────────────────────
@app.post('/clone', tags=['Voice Cloning'])
async def clone_voice(
    text: str = Form(..., description='Text to speak in the cloned voice'),
    language: str = Form(default='en', description='Language code: en, fr, de, es, it, pt, zh, ja, ko, hi, ar, ru'),
    reference_audio: UploadFile = File(..., description='Audio sample of target voice (min 6 seconds)'),
    api_key: str = Security(verify_key)
):
    """
    Clone a voice and generate speech.
    - Accepts: WAV, MP3, M4A, OGG, FLAC, AAC
    - Minimum 6 seconds reference audio
    - Returns: WAV audio file
    """

    # Validate text
    if len(text.strip()) == 0:
        raise HTTPException(status_code=400, detail='Text cannot be empty')
    if len(text) > 5000:
        raise HTTPException(status_code=400, detail='Text too long. Max 5000 characters.')

    # Validate file extension
    filename  = reference_audio.filename.lower()
    allowed   = ['.wav', '.mp3', '.m4a', '.ogg', '.flac', '.aac']
    if not any(filename.endswith(ext) for ext in allowed):
        raise HTTPException(
            status_code=400,
            detail=f'Unsupported format. Allowed: {allowed}'
        )

    # Save uploaded file
    ref_path    = f'uploads/ref_{reference_audio.filename}'
    clean_path  = f'uploads/clean_{reference_audio.filename}.wav'
    output_path = f'outputs/cloned_{reference_audio.filename.split(".")[0]}.wav'

    with open(ref_path, 'wb') as f:
        f.write(await reference_audio.read())

    # Convert to WAV if needed
    if not ref_path.lower().endswith('.wav'):
        converted_path = ref_path.rsplit('.', 1)[0] + '.wav'
        convert_to_wav(ref_path, converted_path)
        ref_path = converted_path

    # Check quality
    quality = score_audio_quality(ref_path)
    if quality < 20:
        raise HTTPException(
            status_code=400,
            detail=f'Audio quality too low ({quality}/100). Use a cleaner recording.'
        )

    # Clean audio
    clean_audio(ref_path, clean_path)

    # Generate cloned speech
    tts.tts_to_file(
        text=text,
        speaker_wav=clean_path,
        language=language,
        file_path=output_path
    )

    return FileResponse(
        output_path,
        media_type='audio/wav',
        filename='cloned_voice.wav',
        headers={'X-Quality-Score': str(quality)}
    )

print('✓ FastAPI app built successfully')
print('✓ Endpoints: /, /health, /check-audio, /clone')

---
## Cell 7 — Start Server + Get Public URL

In [ ]:
import uvicorn
import nest_asyncio
import threading
import subprocess
import time
import os
from pyngrok import ngrok

nest_asyncio.apply()

# Kill port 8000 if already in use
subprocess.run(['fuser', '-k', '8000/tcp'], capture_output=True)
time.sleep(2)

# Kill existing ngrok tunnels
ngrok.kill()
time.sleep(1)

# Create public tunnel
tunnel     = ngrok.connect(8000)
public_url = tunnel.public_url

print('=' * 55)
print(f"  🚀 API is LIVE")
print('=' * 55)
print(f"  Public URL : {public_url}")
print(f"  Docs       : {public_url}/docs")
print(f"  Health     : {public_url}/health")
print(f"  API Key    : {API_KEY}")
print('=' * 55)

# Run server in background thread
config = uvicorn.Config(
    app,
    host='0.0.0.0',
    port=8000,
    log_level='warning'
)
server = uvicorn.Server(config)
thread = threading.Thread(target=server.run)
thread.daemon = True
thread.start()

time.sleep(3)  # Wait for server to fully start

# Save URL to environment for other cells
os.environ['API_URL'] = public_url

print("✓ Server running in background")
print("✓ You can now run other cells freely")
print(f"\n✓ URL saved — other cells will use it automatically")

---
## Cell 8 — Check API Status
> Run this anytime to verify your API is alive

In [ ]:
import requests
import os
from pyngrok import ngrok

def check_api_status():
    print('=' * 50)
    print('  🔍 API STATUS CHECK')
    print('=' * 50)

    # Check tunnel
    print('\n1️⃣  Checking ngrok tunnel...')
    tunnels = ngrok.get_tunnels()
    if not tunnels:
        print('  ✗ No active tunnels — rerun Cell 7')
        return
    url = tunnels[0].public_url
    print(f'  ✓ Tunnel active: {url}')

    # Ping server
    print('\n2️⃣  Pinging server...')
    try:
        res = requests.get(
            f'{url}/health',
            headers={'X-API-Key': API_KEY},
            timeout=10
        )
        if res.status_code == 200:
            data = res.json()
            print(f"  ✓ Server is LIVE")
            print(f"  ✓ Model  : {data.get('model')}")
            print(f"  ✓ Device : {data.get('device')}")
            print(f"  ✓ GPU    : {data.get('gpu_available')}")
        else:
            print(f'  ✗ Server returned: {res.status_code}')
    except requests.exceptions.ConnectionError:
        print('  ✗ Cannot reach server — rerun Cell 7')
        return
    except requests.exceptions.Timeout:
        print('  ✗ Server timed out')
        return

    # Check API key
    print('\n3️⃣  Checking API key security...')
    res = requests.get(
        f'{url}/health',
        headers={'X-API-Key': 'wrong-key'},
        timeout=10
    )
    if res.status_code == 403:
        print('  ✓ API key security working')
    else:
        print('  ⚠️  API key security may not be active')

    # Check docs
    print('\n4️⃣  Checking docs page...')
    res = requests.get(f'{url}/docs', timeout=10)
    if res.status_code == 200:
        print('  ✓ Docs accessible')
    else:
        print(f'  ✗ Docs returned: {res.status_code}')

    # Summary
    print('\n' + '=' * 50)
    print('  📋 QUICK REFERENCE')
    print('=' * 50)
    print(f'  URL    : {url}')
    print(f'  Docs   : {url}/docs')
    print(f'  Key    : {API_KEY}')
    print('=' * 50)
    print('\n  📋 Copy-paste for your software:\n')
    print(f'  BASE_URL = "{url}"')
    print(f'  API_KEY  = "{API_KEY}"')
    print(f'  headers  = {{"X-API-Key": "{API_KEY}"}}')
    print('=' * 50)

check_api_status()

---
## Cell 9 — Test Voice Cloning
> Upload your audio file to Colab files panel first

In [ ]:
import requests
import os
import time
from IPython.display import Audio, display

BASE_URL = os.environ.get('API_URL')
headers  = {'X-API-Key': API_KEY}

# ⚠️ Change this to your actual audio filename
AUDIO_FILE = 'your_audio.m4a'
TEXT       = 'Hello this is my cloned voice speaking through the API.'
LANGUAGE   = 'en'

print(f'Testing: {BASE_URL}\n')

# ── Step 1: Health check ───────────────────────
print('1️⃣  Health check...')
try:
    res = requests.get(f'{BASE_URL}/health', headers=headers, timeout=10)
    print(f'  ✓ {res.json()}\n')
except Exception as e:
    print(f'  ✗ Failed: {e}')
    print('  → Rerun Cell 7 and wait 10 seconds')

# ── Step 2: Audio quality check ────────────────
print('2️⃣  Checking audio quality...')
try:
    with open(AUDIO_FILE, 'rb') as audio:
        res = requests.post(
            f'{BASE_URL}/check-audio',
            headers=headers,
            files={'reference_audio': (AUDIO_FILE, audio, 'audio/m4a')},
            timeout=30
        )
    data = res.json()
    print(f"  ✓ Score    : {data.get('quality_score')}/100")
    print(f"  ✓ Rating   : {data.get('rating')}")
    print(f"  ✓ Duration : {data.get('duration_seconds')}s")
    print(f"  ✓ Advice   : {data.get('recommendation')}\n")
except Exception as e:
    print(f'  ✗ Failed: {e}\n')

# ── Step 3: Clone voice ────────────────────────
print('3️⃣  Cloning voice (20-40 seconds on GPU)...')
try:
    with open(AUDIO_FILE, 'rb') as audio:
        res = requests.post(
            f'{BASE_URL}/clone',
            headers=headers,
            data={'text': TEXT, 'language': LANGUAGE},
            files={'reference_audio': (AUDIO_FILE, audio, 'audio/m4a')},
            timeout=120
        )

    if res.status_code == 200:
        with open('result.wav', 'wb') as f:
            f.write(res.content)
        print('  ✓ Cloned audio saved as result.wav')
        print('  ▶ Playing audio...')
        display(Audio('result.wav'))
    else:
        try:
            print(f'  ✗ Error: {res.json()}')
        except:
            print(f'  ✗ Status: {res.status_code}')
            print(f'  ✗ Response: {res.text[:500]}')

except requests.exceptions.Timeout:
    print('  ✗ Timed out — try again or increase timeout')
except Exception as e:
    print(f'  ✗ Failed: {e}')

---
## Cell 10 — Get Connection Details
> Run anytime you need your URL and key

In [ ]:
import os
from pyngrok import ngrok

print('=' * 50)
print('  📡 API CONNECTION DETAILS')
print('=' * 50)

tunnels = ngrok.get_tunnels()

if tunnels:
    url = tunnels[0].public_url
    print(f'  Server URL  : {url}')
    print(f'  Docs        : {url}/docs')
    print(f'  Health      : {url}/health')
    print(f'  Clone       : {url}/clone')
    print(f'  Check Audio : {url}/check-audio')
else:
    print('  ✗ No active tunnels — rerun Cell 7')
    url = 'NOT ACTIVE'

print(f'  API Key     : {API_KEY}')
print('=' * 50)
print('\n  📋 Copy-paste into your software:\n')
print(f'  BASE_URL = "{url}"')
print(f'  API_KEY  = "{API_KEY}"')
print(f'  headers  = {{"X-API-Key": "{API_KEY}"}}')
print('=' * 50)

---
## Cell 11 — Save Outputs to Google Drive
> Run this to persist your cloned audio files between sessions

In [ ]:
from google.colab import drive
import shutil
import os

drive.mount('/content/drive')

drive_output = '/content/drive/MyDrive/voice_cloning_outputs'
os.makedirs(drive_output, exist_ok=True)

files = os.listdir('outputs')

if files:
    for f in files:
        shutil.copy(f'outputs/{f}', f'{drive_output}/{f}')
        print(f'  ✓ Saved: {f}')
    print(f'\n✓ All outputs saved to: {drive_output}')
else:
    print('  No output files found yet — run Cell 9 first to clone a voice')

---
## 📖 Quick Reference

### Calling From Your Software
```python
import requests

BASE_URL = 'https://YOUR-NGROK-URL.ngrok-free.dev'
headers  = {'X-API-Key': 'YOUR_API_KEY'}

with open('reference.m4a', 'rb') as audio:
    res = requests.post(
        f'{BASE_URL}/clone',
        headers=headers,
        data={'text': 'Hello world', 'language': 'en'},
        files={'reference_audio': audio},
        timeout=120
    )

with open('output.wav', 'wb') as f:
    f.write(res.content)
```

### Supported Languages
| Code | Language | Code | Language |
|------|----------|------|----------|
| en | English | fr | French |
| de | German | es | Spanish |
| it | Italian | pt | Portuguese |
| zh | Chinese | ja | Japanese |
| ko | Korean | hi | Hindi |
| ar | Arabic | ru | Russian |

### Tips for Best Clone Quality
- Minimum **6 seconds** reference audio (30 seconds = much better)
- Record in a **quiet room** — no background noise or echo
- Use **natural speech** — varied pitch clones better than monotone
- Run `/check-audio` first to score your reference file
- Score above **60** = good clone. Above **80** = excellent.

### Troubleshooting
| Error | Fix |
|-------|-----|
| Tunnel not found | Rerun Cell 7 |
| Port 8000 in use | Cell 7 auto-kills it |
| Model not found | Rerun Cell 4 |
| Variables undefined | Run all cells top to bottom |
| Empty response | Wait 5 seconds after Cell 7 |
| M4A file error | ffmpeg installed in Cell 1 — auto-converts |

---
*Built with XTTS v2 by Coqui TTS | FastAPI | ngrok*